# Tutorial 5: Trajectory Selection

History matching identifies the **NROY region** — the part of parameter space *Not Ruled Out Yet* by the available evidence. But for stochastic models, knowing *which parameters are plausible* is only half the story. The same parameters with different random seeds can produce very different epidemic curves.

**Trajectory selection** is the post-calibration step that picks specific `(parameter set, random seed)` pairs whose simulated outputs are consistent with the observed data. The result is a set of **plausible trajectories** — not just plausible parameters.

This tutorial uses **Sampling Importance Resampling (SIR)**¹ to select trajectories:

1. Sample `(β, γ, seed)` triples from the NROY region
2. Run the stochastic simulator for each triple → get an ensemble of incidence curves
3. Weight each trajectory by how well it matches the observations (pseudo-likelihood)
4. Resample proportional to those weights → the selected trajectory set

---
¹ *Yes, the acronym matches the epidemic model. This is intentional and delightful.*

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

import historymatching as hm
from model import SIR, generate_observed_data

%matplotlib inline


## 1. Generate synthetic observations

We generate a synthetic epidemic using a known `(β, γ)` pair. In a real study this would be replaced by actual surveillance data.

In [ ]:
POPULATION       = 5_000
SEED_INFECTIONS  = 50
BETA_TRUE        = 0.7
GAMMA_TRUE       = 0.4

obs_incidence, obs_model = generate_observed_data(
    beta_true        = BETA_TRUE,
    gamma_true       = GAMMA_TRUE,
    population_size  = POPULATION,
    n_seed_infections= SEED_INFECTIONS,
    seed=40,
)

print(f"True parameters:  β={BETA_TRUE}, γ={GAMMA_TRUE}  (R₀ ≈ {BETA_TRUE/GAMMA_TRUE:.2f})")
print(f"Observed peak:    {obs_incidence.max():.0f} cases on day {obs_incidence.argmax()}")
print(f"Total cases:      {obs_incidence.sum():.0f}  (attack rate {obs_incidence.sum()/POPULATION:.1%})")

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(obs_incidence.values, 'ko-', ms=4, lw=1.5, label='Observed incidence')
ax.set_xlabel('Day')
ax.set_ylabel('Daily new cases')
ax.set_title('Observed epidemic curve')
ax.legend()
ax.grid(ls=':')
fig.tight_layout()

## 2. Run history matching to identify the NROY region

We calibrate using three summary statistics — peak incidence, attack rate, and incidence at day 5.

The simulation function **injects a `rand_seed`** column into the samples DataFrame if one isn't already present. Each `(β, γ, seed)` triple then maps to a single deterministic output, giving the GPR emulator a clean surface to fit. The engine ignores `rand_seed` for emulation (it only passes parameter-space columns to the emulator), but stores it on `result.samples` so the exact triples are available for trajectory selection later.

In [ ]:
def run_sir(samples: pd.DataFrame) -> pd.DataFrame:
    """Simulator wrapper — injects rand_seed for reproducibility.

    The engine only sees 'beta' and 'gamma' as parameter-space columns.
    'rand_seed' is metadata that flows through on result.samples.
    """
    df = samples.copy()
    if 'rand_seed' not in df.columns:
        df['rand_seed'] = np.random.default_rng(42).integers(0, 2**31, size=len(df))

    rows = []
    for _, row in df.iterrows():
        model = SIR(
            beta  = row['beta'],
            gamma = row['gamma'],
            s0    = POPULATION - SEED_INFECTIONS,
            i0    = SEED_INFECTIONS,
            seed  = int(row['rand_seed']),
        )
        inc = model.get_incidence()
        rows.append({
            'peak_incidence': float(inc.max()),
            'attack_rate':    float(inc.sum() / POPULATION),
            'incidence_5':    float(inc[5]) if len(inc) > 5 else 0.0,
        })
    return pd.DataFrame(rows)

In [ ]:
obs_peak  = float(obs_incidence.max())
obs_ar    = float(obs_incidence.sum() / POPULATION)
obs_inc5  = float(obs_incidence.values[5])

print(f"Observed peak:        {obs_peak:.0f} cases/day")
print(f"Observed incidence_5: {obs_inc5:.0f} cases/day")
print(f"Observed attack rate: {obs_ar:.2f}")

builder = hm.HistoryMatchingBuilder.from_data(
    parameter_bounds = {'beta': (0.3, 2.0), 'gamma': (0.1, 0.8)},
    observations     = {
        'peak_incidence': (obs_peak,  obs_peak * 0.05),   # ±5%
        'attack_rate':    (obs_ar,    obs_ar   * 0.03),   # ±3%
        'incidence_5':    (obs_inc5,  obs_inc5 * 0.10),   # ±10%
    },
)
builder.emulator_type = 'gpr'
builder.n_samples = 500
builder.max_iterations = 5
builder.implausibility_threshold = 3.0
engine = builder.build()
engine.set_simulation_function(run_sir)

results = engine.run()
print(engine)

## 3. Inspect the NROY region

After history matching the parameter samples from the final iteration were drawn by rejection-sampling the full prior against all trained emulators. They represent a draw from the current NROY region.

In [ ]:
# Collect samples from all iterations to visualise the space reduction
all_samples = pd.concat(
    [r.samples.assign(iteration=r.iteration) for r in results],
    ignore_index=True,
)

fig, axs = plt.subplots(1, 2, figsize=(12, 4))

cmap  = plt.get_cmap('viridis')
n_iter = len(results)
for it in range(1, n_iter + 1):
    df = all_samples[all_samples['iteration'] == it]
    axs[0].scatter(df['beta'], df['gamma'],
                   s=10, alpha=0.5,
                   color=cmap((it - 1) / n_iter),
                   label=f'Iteration {it}')

axs[0].scatter([BETA_TRUE], [GAMMA_TRUE], marker='*', s=200,
               c='red', zorder=5, label='True values')
axs[0].set_xlabel('β')
axs[0].set_ylabel('γ')
axs[0].set_title('Parameter space — NROY shrinks each iteration')
axs[0].legend(fontsize=8)
axs[0].grid(ls=':')

# NROY fraction from the engine (what fraction of the prior remains plausible)
iters = [r.iteration for r in results]
nroy_fracs = [r.nroy_fraction for r in results]

axs[1].bar(iters, nroy_fracs, color=[cmap((i) / n_iter) for i in range(n_iter)])
for w, f in zip(iters, nroy_fracs):
    axs[1].annotate(f'{f:.0%}', (w, f), textcoords='offset points',
                    xytext=(0, 5), ha='center', fontsize=9)
axs[1].set_xlabel('Iteration')
axs[1].set_ylabel('NROY fraction')
axs[1].set_title('Fraction of prior space remaining')
axs[1].set_xticks(iters)
axs[1].set_ylim(0, 1)
axs[1].grid(ls=':', axis='y')

fig.tight_layout()

# NROY pool = samples that passed ALL waves' emulators
# These already have rand_seed attached (injected by run_sir)
nroy_samples = engine.get_nroy_samples(10000)
print(f"NROY parameter pool: {len(nroy_samples)} (β, γ, seed) triples")
print(f"  β range:    [{nroy_samples['beta'].min():.3f}, {nroy_samples['beta'].max():.3f}]")
print(f"  γ range:    [{nroy_samples['gamma'].min():.3f}, {nroy_samples['gamma'].max():.3f}]")


## 4. Generate a trajectory ensemble

We draw a large NROY pool using `engine.get_nroy_samples(10000)` — this rejection-samples fresh candidates from the full prior against all trained emulators, cheaply and without running any new simulations. Each NROY point gets a unique seed so the trajectory is deterministic and reproducible.


In [ ]:
N_DAYS = obs_incidence.shape[0]

ensemble = []
for i, (_, row) in enumerate(nroy_samples.iterrows()):
    model = SIR(
        beta  = row['beta'],
        gamma = row['gamma'],
        s0    = POPULATION - SEED_INFECTIONS,
        i0    = SEED_INFECTIONS,
        seed  = i,   # unique seed per NROY point
    )
    inc = model.get_incidence()
    ensemble.append({
        'beta':      row['beta'],
        'gamma':     row['gamma'],
        'seed':      i,
        'incidence': inc[:N_DAYS],
    })

ensemble_df = pd.DataFrame(ensemble)
print(f"Ensemble size: {len(ensemble_df)} trajectories (1 per NROY sample)")

In [ ]:
inc_matrix = np.vstack(ensemble_df['incidence'].values)  # shape (n_traj, n_days)

# Compute calibration features for every trajectory
ensemble_df['peak']  = inc_matrix.max(axis=1)
ensemble_df['ar']    = inc_matrix.sum(axis=1) / POPULATION
ensemble_df['inc5']  = inc_matrix[:, 5]

fig, axs = plt.subplots(2, 2, figsize=(14, 8))

# Top-left: time-series spaghetti with day 5 highlighted
ax = axs[0, 0]
for row in inc_matrix:
    ax.plot(row, color='steelblue', alpha=0.05, lw=0.8)
ax.plot(inc_matrix.mean(axis=0), color='steelblue', lw=2, label='Ensemble mean')
ax.plot(obs_incidence.values, 'ko-', ms=4, lw=2, label='Observed', zorder=5)
ax.scatter([5], [obs_inc5], s=120, facecolors='none', edgecolors='red',
           linewidths=2, zorder=6, label='Emulated day 5')
ax.set_xlabel('Day')
ax.set_ylabel('Daily new cases')
ax.set_title(f'Prior NROY ensemble ({len(ensemble_df):,} trajectories)')
ax.legend(fontsize=8)
ax.grid(ls=':')

# Top-right: peak incidence
ax = axs[0, 1]
ax.hist(ensemble_df['peak'], bins=30, color='steelblue', alpha=0.6, edgecolor='white')
ax.axvline(obs_peak, color='red', lw=2, ls='--', label=f'Observed ({obs_peak:.0f})')
ax.axvspan(obs_peak * (1 - 1.96 * 0.05), obs_peak * (1 + 1.96 * 0.05),
           color='red', alpha=0.08, label='HM tolerance (±5%)')
ax.set_xlabel('Peak incidence')
ax.set_ylabel('Count')
ax.set_title('Peak incidence')
ax.legend(fontsize=8)
ax.grid(ls=':')

# Bottom-left: attack rate
ax = axs[1, 0]
ax.hist(ensemble_df['ar'], bins=30, color='steelblue', alpha=0.6, edgecolor='white')
ax.axvline(obs_ar, color='red', lw=2, ls='--', label=f'Observed ({obs_ar:.2f})')
ax.axvspan(obs_ar * (1 - 1.96 * 0.03), obs_ar * (1 + 1.96 * 0.03),
           color='red', alpha=0.08, label='HM tolerance (±3%)')
ax.set_xlabel('Attack rate')
ax.set_ylabel('Count')
ax.set_title('Attack rate')
ax.legend(fontsize=8)
ax.grid(ls=':')

# Bottom-right: incidence day 5
ax = axs[1, 1]
ax.hist(ensemble_df['inc5'], bins=30, color='steelblue', alpha=0.6, edgecolor='white')
ax.axvline(obs_inc5, color='red', lw=2, ls='--', label=f'Observed ({obs_inc5:.0f})')
ax.axvspan(obs_inc5 * (1 - 1.96 * 0.10), obs_inc5 * (1 + 1.96 * 0.10),
           color='red', alpha=0.08, label='HM tolerance (±10%)')
ax.set_xlabel('Incidence day 5')
ax.set_ylabel('Count')
ax.set_title('Incidence day 5')
ax.legend(fontsize=8)
ax.grid(ls=':')

fig.suptitle('Prior NROY ensemble vs HM targets', fontsize=12)
fig.tight_layout()

## 5. Weight trajectories by pseudo-likelihood

We score each trajectory against the observed incidence curve using a **Gaussian pseudo-likelihood**:

$$w_i = \exp\!\left(-\frac{d_i^2}{2\,\varepsilon^2}\right), \qquad d_i = \sqrt{\frac{1}{T}\sum_{t=1}^T\left(y_i(t) - y_{\mathrm{obs}}(t)\right)^2}$$

where $d_i$ is the RMSE between trajectory $i$ and the observed data, and $\varepsilon$ is a **tolerance** that controls how strict the selection is.

- **Large ε** → lenient: weights spread broadly, selected set ≈ random draw from NROY  
- **Small ε** → strict: only trajectories very close to the observed curve get non-negligible weight

A practical starting point is $\varepsilon \approx$ one standard deviation of the observed peak.

In [ ]:
def gaussian_weights(inc_matrix: np.ndarray, obs: np.ndarray, epsilon: float) -> np.ndarray:
    """
    Compute normalised pseudo-likelihood weights for each trajectory.

    Args:
        inc_matrix: Array of shape (n_traj, n_days) — simulated trajectories.
        obs:        Array of shape (n_days,)         — observed incidence.
        epsilon:    Tolerance (RMSE scale).  Smaller = stricter selection.

    Returns:
        Normalised weight array of shape (n_traj,).
    """
    rmse = np.sqrt(np.mean((inc_matrix - obs[np.newaxis, :]) ** 2, axis=1))
    w    = np.exp(-0.5 * (rmse / epsilon) ** 2)
    total = w.sum()
    if total == 0:
        raise ValueError("All weights are zero — try a larger epsilon.")
    return w / total


# Tolerance ≈ 10 % of the peak value — a reasonable starting point
epsilon = obs_peak * 0.03

obs_array = obs_incidence.values[:N_DAYS]
weights   = gaussian_weights(inc_matrix, obs_array, epsilon=epsilon)

print(f"Tolerance ε = {epsilon:.1f} cases/day")
print(f"Effective sample size: {1 / (weights**2).sum():.0f}  "
      f"(out of {len(weights):,} total)")

# Scatter: parameter space coloured by weight
fig, ax = plt.subplots(figsize=(6, 5))
sc = ax.scatter(
    ensemble_df['beta'], ensemble_df['gamma'],
    c=weights, cmap='viridis_r', s=6, alpha=0.6, norm=mcolors.LogNorm(),
)
ax.scatter([BETA_TRUE], [GAMMA_TRUE], marker='*', s=250, c='red',
           edgecolors='k', lw=0.5, zorder=5, label='True values')
plt.colorbar(sc, ax=ax, label='Weight (log scale)')
ax.set_xlabel('β')
ax.set_ylabel('γ')
ax.set_title('Trajectory weights in parameter space')
ax.legend()
ax.grid(ls=':')
fig.tight_layout()

## 6. Importance resampling — select the final trajectory set

We draw `N_SELECT` trajectories without replacement from the ensemble, proportional to their weights. The result is a **volume-manageable set of plausible trajectories** that can be used for downstream analysis (forecasting quantiles, reporting representative epidemic curves, etc.).

In [ ]:
N_SELECT  = 15   # number of trajectories to retain
rng       = np.random.default_rng(seed=0)

selected_idx = rng.choice(
    len(ensemble_df),
    size    = N_SELECT,
    replace = False,
    p       = weights,
)
selected_df  = ensemble_df.iloc[selected_idx].copy()
selected_inc = inc_matrix[selected_idx]

print(f"Selected {N_SELECT} trajectories")
print(f"β range in selected set:  {selected_df['beta'].min():.3f} – {selected_df['beta'].max():.3f}")
print(f"γ range in selected set:  {selected_df['gamma'].min():.3f} – {selected_df['gamma'].max():.3f}")

# ── Consistency check: re-run selected triples, verify identical outputs ──
print("\nReproducibility check (re-running 5 selected trajectories):")
for check_idx in selected_idx[:5]:
    row = ensemble_df.iloc[check_idx]
    model = SIR(
        beta  = row['beta'],
        gamma = row['gamma'],
        s0    = POPULATION - SEED_INFECTIONS,
        i0    = SEED_INFECTIONS,
        seed  = int(row['seed']),
    )
    rerun_inc = model.get_incidence()[:N_DAYS]
    orig_inc  = inc_matrix[check_idx]
    match = np.allclose(rerun_inc, orig_inc)
    print(f"  idx={check_idx:4d}  β={row['beta']:.3f}  γ={row['gamma']:.3f}  "
          f"seed={int(row['seed'])}  match={match}")
    assert match, f"Trajectory {check_idx} not reproducible!"

print("All checks passed — selected trajectories are reproducible.")

## 7. Visualise the selected trajectories

Compare the full prior ensemble (grey) against the selected trajectories (coloured) and the observations (black). The selected set should be visually consistent with the observed curve, whereas the full ensemble is much broader.

In [ ]:
selected_peaks = selected_inc.max(axis=1)
selected_ar    = selected_inc.sum(axis=1) / POPULATION
selected_inc5  = selected_inc[:, 5]

fig, axs = plt.subplots(2, 3, figsize=(17, 10))

# ── Top row: time-series comparison ──────────────────────────────────────

# Top-left: full prior ensemble
ax = axs[0, 0]
for row in inc_matrix:
    ax.plot(row, color='steelblue', alpha=0.04, lw=0.7)
ax.plot(obs_array, 'ko-', ms=4, lw=2, label='Observed', zorder=5)
ax.scatter([5], [obs_inc5], s=120, facecolors='none', edgecolors='red',
           linewidths=2, zorder=6, label='Emulated day 5')
ax.set_title(f'Prior ensemble\n({len(inc_matrix):,} trajectories)')
ax.set_xlabel('Day')
ax.set_ylabel('Daily new cases')
ax.legend(fontsize=8)
ax.grid(ls=':')

# Top-middle: selected trajectories
ax = axs[0, 1]
cmap_sel = plt.get_cmap('plasma')
for k, row in enumerate(selected_inc):
    ax.plot(row, color=cmap_sel(k / N_SELECT), alpha=0.6, lw=1.0)
ax.fill_between(
    range(N_DAYS),
    selected_inc.min(axis=0),
    selected_inc.max(axis=0),
    alpha=0.15, color='purple', label='Selected range',
)
ax.plot(selected_inc.mean(axis=0), color='purple', lw=2.5, label='Selected mean')
ax.plot(obs_array, 'ko-', ms=4, lw=2, label='Observed', zorder=5)
ax.scatter([5], [obs_inc5], s=120, facecolors='none', edgecolors='red',
           linewidths=2, zorder=6, label='Emulated day 5')
ax.set_title(f'Selected trajectories\n({N_SELECT} via importance resampling)')
ax.set_xlabel('Day')
ax.legend(fontsize=8)
ax.grid(ls=':')

# Top-right: parameter space
ax = axs[0, 2]
ax.scatter(ensemble_df['beta'], ensemble_df['gamma'],
           s=4, alpha=0.2, color='steelblue', label='Prior NROY')
ax.scatter(selected_df['beta'], selected_df['gamma'],
           s=20, alpha=0.8, color='purple', label='Selected')
ax.scatter([BETA_TRUE], [GAMMA_TRUE], marker='*', s=200,
           c='red', zorder=5, label='True values')
ax.set_xlabel('β')
ax.set_ylabel('γ')
ax.set_title('Parameter space')
ax.legend(fontsize=8)
ax.grid(ls=':')

# ── Bottom row: feature-space comparison ─────────────────────────────────

# Bottom-left: peak incidence
ax = axs[1, 0]
ax.hist(ensemble_df['peak'], bins=30, density=True, color='steelblue',
        alpha=0.4, edgecolor='white', label='Prior NROY')
ax.hist(selected_peaks, bins=15, density=True, color='purple',
        alpha=0.7, edgecolor='white', label='Selected')
ax.axvline(obs_peak, color='red', lw=2, ls='--', label=f'Observed ({obs_peak:.0f})')
ax.set_xlabel('Peak incidence')
ax.set_ylabel('Density')
ax.set_title('Peak incidence')
ax.legend(fontsize=8)
ax.grid(ls=':')

# Bottom-middle: attack rate
ax = axs[1, 1]
ax.hist(ensemble_df['ar'], bins=30, density=True, color='steelblue',
        alpha=0.4, edgecolor='white', label='Prior NROY')
ax.hist(selected_ar, bins=15, density=True, color='purple',
        alpha=0.7, edgecolor='white', label='Selected')
ax.axvline(obs_ar, color='red', lw=2, ls='--', label=f'Observed ({obs_ar:.2f})')
ax.set_xlabel('Attack rate')
ax.set_ylabel('Density')
ax.set_title('Attack rate')
ax.legend(fontsize=8)
ax.grid(ls=':')

# Bottom-right: incidence day 5
ax = axs[1, 2]
ax.hist(ensemble_df['inc5'], bins=30, density=True, color='steelblue',
        alpha=0.4, edgecolor='white', label='Prior NROY')
ax.hist(selected_inc5, bins=15, density=True, color='purple',
        alpha=0.7, edgecolor='white', label='Selected')
ax.axvline(obs_inc5, color='red', lw=2, ls='--', label=f'Observed ({obs_inc5:.0f})')
ax.set_xlabel('Incidence day 5')
ax.set_ylabel('Density')
ax.set_title('Incidence day 5')
ax.legend(fontsize=8)
ax.grid(ls=':')

fig.suptitle('NROY ensemble  →  Trajectory selection', fontsize=13)
fig.tight_layout()

In [ ]:
# Parameter distributions: NROY pool vs. selected set
fig, axs = plt.subplots(1, 2, figsize=(11, 4))

for ax, param, true_val in zip(axs, ['beta', 'gamma'], [BETA_TRUE, GAMMA_TRUE]):
    ax.hist(nroy_samples[param], bins=20, density=True,
            alpha=0.5, color='steelblue', label='NROY pool')
    ax.hist(selected_df[param], bins=20, density=True,
            alpha=0.7, color='purple', label='Selected')
    ax.axvline(true_val, color='red', ls='--', lw=1.5, label=f'True ({true_val})')
    ax.set_xlabel(param)
    ax.set_ylabel('Density')
    ax.set_title(f'Distribution of {param}')
    ax.legend(fontsize=8)
    ax.grid(ls=':')

fig.suptitle('Parameter distributions before and after trajectory selection', fontsize=12)
fig.tight_layout()

## 8. Effect of tolerance ε

The tolerance `ε` is the key tuning parameter. Here we sweep over several values to show the trade-off between **specificity** (small ε → selected trajectories are close to the observed data) and **coverage** (large ε → selected trajectories span the full NROY ensemble).

A useful diagnostic is the **effective sample size** $N_{\mathrm{eff}} = \left(\sum_i w_i^2\right)^{-1}$. When $N_{\mathrm{eff}} \ll N$ the weight distribution is too concentrated; when $N_{\mathrm{eff}} \approx N$ the selection is almost uniform (lenient).

In [ ]:
epsilons = [obs_peak * f for f in [0.02, 0.05, 0.10, 0.20, 0.40]]

fig, axs = plt.subplots(1, len(epsilons), figsize=(16, 4), sharey=True)

for ax, eps in zip(axs, epsilons):
    w   = gaussian_weights(inc_matrix, obs_array, epsilon=eps)
    nef = 1 / (w**2).sum()

    idx_sel = rng.choice(len(ensemble_df), size=N_SELECT, replace=False, p=w)
    sel_inc = inc_matrix[idx_sel]

    ax.fill_between(
        range(N_DAYS),
        np.percentile(sel_inc, 10, axis=0),
        np.percentile(sel_inc, 90, axis=0),
        alpha=0.3, color='purple', label='10–90 pct',
    )
    ax.plot(sel_inc.mean(axis=0), color='purple', lw=2)
    ax.plot(obs_array, 'ko-', ms=3, lw=1.5, zorder=5)
    ax.set_title(f'ε = {eps:.1f}\nNeff = {nef:.0f}')
    ax.set_xlabel('Day')
    ax.grid(ls=':')

axs[0].set_ylabel('Daily new cases')
fig.suptitle('Trajectory selection sensitivity to tolerance ε', fontsize=12)
fig.tight_layout()

## Summary

| Step | What it does | Output |
|------|-------------|--------|
| History matching | Eliminates implausible regions of parameter space using emulators | NROY parameter samples |
| Trajectory ensemble | Runs the stochastic model at NROY parameters × multiple seeds | Large ensemble of candidate trajectories |
| Pseudo-likelihood weighting | Scores each trajectory by how closely it matches the observed data | Weight per `(parameter, seed)` pair |
| Importance resampling | Draws a tractable set proportional to weights | Selected plausible trajectories |

### Key design choices

**Why use summary statistics for HM but full time series for trajectory selection?**  
History matching with the full time series would require one emulator per day — high-dimensional and expensive. Summary statistics keep HM tractable. The trajectory selection step then acts as a second, finer filter using the full data.

**How to choose ε?**  
Start with ε ≈ 10% of the observed peak. Check the effective sample size — aim for $N_{\mathrm{eff}} \gtrsim 50$. If the selected trajectories look too broad, tighten ε. If they look implausibly narrow, relax it.

**Using the selected trajectories**  
Each selected `(β, γ, seed)` triple is a complete, reproducible simulation run. You can re-run these with the full model (e.g., longer time horizon, additional compartments) to generate matched forecasts or counterfactuals.